# EWoK score columns + material-dynamics concept comparison

This notebook uses minimal adaptations of `ewok_eval.py` logic to compute per-item columns:

- `S11 = score(T1 | C1)`
- `S12 = score(T2 | C1)`
- `S22 = score(T2 | C2)`
- `S21 = score(T1 | C2)`

Combined metrics included:

- Margin-based (from `ewok_eval.py`):
  - `margin_combined = 0.5 * ((S11 - S12) + (S22 - S21))`
  - `correct_combined = (margin_combined > 0)`
- Boolean-pair (requested):
  - `b1 = 1[(S11 - S12) > 0]`
  - `b2 = 1[(S22 - S21) > 0]`
  - `combined_boolean_pair = 0.5 * (b1 + b2)` in `{0.0, 0.5, 1.0}`

Then it focuses on `Domain == material-dynamics` and compares per-concept scores.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

# Run this notebook from the moonshotGPT repo root, or adjust PROJECT_ROOT manually.
PROJECT_ROOT = Path.cwd().resolve()
EXPERIMENT_DIR = PROJECT_ROOT / "runs" / "research" / "bos_aligned_proto" / "<run_name>"
CHECKPOINT_DIR = EXPERIMENT_DIR / "ckpt_periodic_step0018000"

BATCH_SIZE = 8
SCORE_REDUCTION = "mean"  # "sum" or "mean"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"checkpoint: {CHECKPOINT_DIR}")
print(f"device: {DEVICE}")
print(f"score_reduction: {SCORE_REDUCTION}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from evaluation import ewok as ewok_eval


In [ ]:
# Optional: if model load fails due to flash-attn binary mismatch, set this to True.
DISABLE_FLASH_ATTN = False

if DISABLE_FLASH_ATTN:
    import transformers.utils.import_utils as iu
    _orig_is_package_available = iu._is_package_available

    def _patched_is_package_available(pkg_name, return_version=False):
        if pkg_name == 'flash_attn':
            return (False, '0.0.0') if return_version else False
        return _orig_is_package_available(pkg_name, return_version=return_version)

    iu._is_package_available = _patched_is_package_available

from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer/model from the requested checkpoint.
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.bos_token

torch_dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
model = AutoModelForCausalLM.from_pretrained(CHECKPOINT_DIR, local_files_only=True, torch_dtype=torch_dtype)
model.eval().to(DEVICE)

print('model loaded')


In [ ]:
# Minimal adaptation of ewok_eval.py logic:
# we reuse ewok_eval.per_token_conditional_log_likelihood and ewok_eval._reduce_token_logps,
# and only add a dataframe-return wrapper with score columns.

def compute_ewok_score_columns(ewok_df, model, tokenizer, batch_size=8, score_reduction='mean', device='cuda'):
    score_reduction = ewok_eval._validate_score_reduction(score_reduction)
    records = []

    for domain in ewok_df['Domain'].unique():
        df_domain = ewok_df[ewok_df['Domain'] == domain].reset_index()

        ctx1 = df_domain['Context1'].tolist()
        tgt1 = df_domain['Target1'].tolist()
        ctx2 = df_domain['Context2'].tolist()
        tgt2 = df_domain['Target2'].tolist()

        r11 = ewok_eval.per_token_conditional_log_likelihood(model, tokenizer, ctx1, tgt1, device=device, batch_size=batch_size)
        r12 = ewok_eval.per_token_conditional_log_likelihood(model, tokenizer, ctx1, tgt2, device=device, batch_size=batch_size)
        r22 = ewok_eval.per_token_conditional_log_likelihood(model, tokenizer, ctx2, tgt2, device=device, batch_size=batch_size)
        r21 = ewok_eval.per_token_conditional_log_likelihood(model, tokenizer, ctx2, tgt1, device=device, batch_size=batch_size)

        for i in range(len(df_domain)):
            s11 = ewok_eval._reduce_token_logps(r11[i], score_reduction)
            s12 = ewok_eval._reduce_token_logps(r12[i], score_reduction)
            s22 = ewok_eval._reduce_token_logps(r22[i], score_reduction)
            s21 = ewok_eval._reduce_token_logps(r21[i], score_reduction)

            m1 = s11 - s12
            m2 = s22 - s21
            m = 0.5 * (m1 + m2)

            # Boolean-pair alternative: average of side-level correctness booleans.
            b1 = bool(m1 > 0)
            b2 = bool(m2 > 0)
            combined_boolean_pair = 0.5 * (float(b1) + float(b2))

            records.append({
                'row_index': int(df_domain.loc[i, 'index']),
                'Domain': domain,
                'score_reduction': score_reduction,
                'S11_score_T1_given_C1': s11,
                'S12_score_T2_given_C1': s12,
                'S22_score_T2_given_C2': s22,
                'S21_score_T1_given_C2': s21,
                'margin_official_m1': m1,
                'margin_symmetric_m2': m2,
                'margin_combined': m,
                'correct_official': b1,
                'correct_symmetric': b2,
                'correct_combined': bool(m > 0),
                'combined_boolean_pair': combined_boolean_pair,
                'both_correct_pair': bool(b1 and b2),
            })

    score_df = pd.DataFrame(records)
    base = ewok_df.reset_index().rename(columns={'index': 'row_index'})
    scored = base.merge(score_df, on=['row_index', 'Domain'], how='left')
    return scored


In [ ]:
ewok_df = ewok_eval.ewok_df.copy()
scored_df = compute_ewok_score_columns(
    ewok_df=ewok_df,
    model=model,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    score_reduction=SCORE_REDUCTION,
    device=DEVICE,
)

cols = [
    'row_index', 'Domain', 'ConceptA', 'ConceptB',
    'S11_score_T1_given_C1', 'S12_score_T2_given_C1',
    'S22_score_T2_given_C2', 'S21_score_T1_given_C2',
    'margin_official_m1', 'margin_symmetric_m2', 'margin_combined',
    'correct_official', 'correct_symmetric', 'correct_combined',
    'combined_boolean_pair', 'both_correct_pair',
]

print('rows:', len(scored_df))
print('overall accuracies/summaries:')
print(scored_df[['correct_official', 'correct_symmetric', 'correct_combined', 'combined_boolean_pair', 'both_correct_pair']].mean())
display(scored_df[cols].head())


In [ ]:
# Material-dynamics: per-concept comparison using both combined metrics
md = scored_df[scored_df['Domain'] == 'material-dynamics'].copy()

concept_long = pd.concat([
    md[['row_index', 'ConceptA', 'correct_combined', 'combined_boolean_pair', 'both_correct_pair', 'margin_combined']].rename(columns={'ConceptA': 'Concept'}),
    md[['row_index', 'ConceptB', 'correct_combined', 'combined_boolean_pair', 'both_correct_pair', 'margin_combined']].rename(columns={'ConceptB': 'Concept'}),
], ignore_index=True)

concept_stats = (
    concept_long.groupby('Concept')
    .agg(
        n=('row_index', 'size'),
        mean_combined_margin_based=('correct_combined', 'mean'),
        mean_combined_boolean_pair=('combined_boolean_pair', 'mean'),
        mean_both_correct_pair=('both_correct_pair', 'mean'),
        mean_margin_combined=('margin_combined', 'mean'),
        mean_abs_margin_combined=('margin_combined', lambda s: s.abs().mean()),
    )
    .sort_values(['mean_combined_boolean_pair', 'n', 'mean_margin_combined'], ascending=[True, False, True])
)

print('material-dynamics rows:', len(md))
display(concept_stats)
display(concept_stats.head(10))  # weakest first by boolean-pair metric


In [ ]:
concept_stats.index

In [ ]:
concept_stats['me']

In [ ]:
# Optional: save outputs for downstream analysis
scores_out = EXPERIMENT_DIR / f'ewok_scored_rows_{SCORE_REDUCTION}_step0018000.csv'
concept_out = EXPERIMENT_DIR / f'ewok_material_dynamics_concept_scores_{SCORE_REDUCTION}_step0018000.csv'

scored_df.to_csv(scores_out, index=False)
concept_stats.reset_index().to_csv(concept_out, index=False)

print('saved:', scores_out)
print('saved:', concept_out)
